# Project: Transfer Learning with Pretrained Models

Use a pretrained MobileNetV2 to classify images without training from scratch.

In [ ]:
import sys, os, subprocess
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_PATH = '/content/drive/MyDrive/data-science-mastery'
    if os.path.isdir(REPO_PATH):
        os.chdir(REPO_PATH)
        print(f'Working directory: {os.getcwd()}')
        if not os.path.isdir('Data') or len(os.listdir('Data')) < 5:
            subprocess.check_call([sys.executable, 'scripts/prepare_datasets.py'])
    else:
        REPO_PATH = '/content/data-science-mastery'
        if os.path.isdir(REPO_PATH):
            os.chdir(REPO_PATH)
        else:
            print('Repo not found. Run opencolab_setup.ipynb first.')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras

%matplotlib inline
print('TensorFlow version:', tf.__version__)

In [ ]:
base = keras.applications.MobileNetV2(
    weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False
model = keras.Sequential([
    base,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(10, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
print('Model created. Base layers frozen.')

## Prepare CIFAR-10 for Transfer Learning

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train, y_test = y_train.flatten(), y_test.flatten()
x_train_resized = tf.image.resize(x_train[:2000], (224, 224)).numpy() / 255.0
x_test_resized = tf.image.resize(x_test[:500], (224, 224)).numpy() / 255.0
y_train, y_test = y_train[:2000], y_test[:500]
class_names = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']
print(f'Train: {x_train_resized.shape}, Test: {x_test_resized.shape}')
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train_resized[i])
    ax.set_title(class_names[y_train[i]])
ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
history = model.fit(x_train_resized, y_train,
                    epochs=5, batch_size=32,
                    validation_data=(x_test_resized, y_test), verbose=1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['accuracy'], label='Train')
axes[0].plot(history.history['val_accuracy'], label='Val')
axes[0].set_title('Accuracy')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['loss'], label='Train')
axes[1].plot(history.history['val_loss'], label='Val')
axes[1].set_title('Loss')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
_, acc = model.evaluate(x_test_resized, y_test, verbose=0)
print(f'Transfer learning test accuracy: {acc:.4f}')

## Summary

- Loaded pretrained MobileNetV2 (ImageNet weights)
- Froze base layers, added custom classifier
- Trained only new top layers on CIFAR-10
- Achieved good accuracy with minimal training